# Onwordly experiment run (Colab)

Runs any `.kaggle-run`-style plan on a Colab GPU. Default: Experiment 001 with the shared format warm-up, compared against the same experiment without it.

Runtime → Change runtime type → **T4 GPU**.

In [ ]:
!git clone https://github.com/HereLiesAz/onwordly.git
%cd onwordly
!pip install -q -e '.[train]'

In [ ]:
RUNS = {
    # name: (experiment, mode, manifest)
    'warmup': ('001', 'single', 'experiments/001-arithmetic-curriculum/manifest-warmup.json'),
    'no-warmup': ('001', 'single', 'experiments/001-arithmetic-curriculum/manifest.json'),
}
SMOKE_FIRST = True  # run the 1k-token smoke manifests first to catch problems in minutes

In [ ]:
import subprocess

def run(name, experiment, mode, manifest):
    plan = f'/content/plan-{name}'
    open(plan, 'w').write(f'experiment: {experiment}\nmode: {mode}\nmanifest: {manifest}\n')
    subprocess.run(['onwordly-kaggle', '--plan', plan, '--output-root', f'/content/results/{name}'], check=True)

if SMOKE_FIRST:
    run('smoke-warmup', '001', 'single', 'experiments/001-arithmetic-curriculum/smoke-manifest-warmup.json')
for name, (experiment, mode, manifest) in RUNS.items():
    run(name, experiment, mode, manifest)

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display
for report in sorted(Path('/content/results').glob('*/*/RESULTS.md')):
    display(Markdown(f'### {report.parent.parent.name}\n\n' + report.read_text()))

In [ ]:
!cd /content && zip -qr results.zip results -x '*.jsonl'
from google.colab import files
files.download('/content/results.zip')